# 05. Analysis & Visualization

Analisis komprehensif hasil semua eksperimen optimasi.

**Contents:**
- Summary semua eksperimen
- Comparison charts
- Trade-off analysis
- Export results untuk paper

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load All Results

In [ ]:
# Load results from experiments
results = {
    'Baseline (ResNet-18)': {
        'accuracy': 93.5,  # Update with actual
        'params_m': 11.17,
        'flops_m': 557.89,
        'latency_ms': 13.78,
        'size_mb': 42.72
    },
    'Pruned (50%)': {
        'accuracy': 89.82,  # Update with actual
        'params_m': 5.58,
        'flops_m': 278.94,
        'latency_ms': 10.5,
        'size_mb': 21.36
    },
    'Distilled (Ghost-ResNet-18)': {
        'accuracy': 94.23,  # Update with actual
        'params_m': 5.77,
        'flops_m': 287.61,
        'latency_ms': 63.46,
        'size_mb': 22.05
    },
    'Quantized (INT8)': {
        'accuracy': 93.2,  # Update with actual
        'params_m': 11.17,
        'flops_m': 557.89,
        'latency_ms': 68.94,
        'size_mb': 10.68
    }
}

df = pd.DataFrame(results).T
df

## 2. Accuracy vs Model Size

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['blue', 'orange', 'green', 'red']
for i, (name, data) in enumerate(results.items()):
    ax.scatter(data['size_mb'], data['accuracy'], s=200, c=colors[i], label=name, alpha=0.7)

ax.set_xlabel('Model Size (MB)', fontsize=14)
ax.set_ylabel('Accuracy (%)', fontsize=14)
ax.set_title('Accuracy vs Model Size Trade-off', fontsize=16)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../weights/accuracy_vs_size.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

models = list(results.keys())
x = np.arange(len(models))
width = 0.6

# Accuracy
axes[0, 0].bar(x, df['accuracy'], width, color=colors)
axes[0, 0].set_ylabel('Accuracy (%)')
axes[0, 0].set_title('Test Accuracy')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models, rotation=15, ha='right')
axes[0, 0].set_ylim([85, 100])

# Parameters
axes[0, 1].bar(x, df['params_m'], width, color=colors)
axes[0, 1].set_ylabel('Parameters (M)')
axes[0, 1].set_title('Model Parameters')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models, rotation=15, ha='right')

# Model Size
axes[1, 0].bar(x, df['size_mb'], width, color=colors)
axes[1, 0].set_ylabel('Size (MB)')
axes[1, 0].set_title('Model Size')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(models, rotation=15, ha='right')

# FLOPs
axes[1, 1].bar(x, df['flops_m'], width, color=colors)
axes[1, 1].set_ylabel('FLOPs (M)')
axes[1, 1].set_title('Computational Cost')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(models, rotation=15, ha='right')

plt.tight_layout()
plt.savefig('../weights/comparison_chart.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Summary Table for Paper

In [ ]:
# Create LaTeX table
baseline = results['Baseline (ResNet-18)']

summary = []
for name, data in results.items():
    summary.append({
        'Method': name,
        'Accuracy (%)': f"{data['accuracy']:.2f}",
        'Params (M)': f"{data['params_m']:.2f}",
        'Size (MB)': f"{data['size_mb']:.2f}",
        'Compression': f"{baseline['size_mb']/data['size_mb']:.2f}x",
        'Acc Drop (%)': f"{baseline['accuracy'] - data['accuracy']:.2f}"
    })

summary_df = pd.DataFrame(summary)
print("\n" + "="*80)
print("SUMMARY TABLE FOR PAPER")
print("="*80)
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv('../weights/experiment_summary.csv', index=False)
print("\nSaved to: weights/experiment_summary.csv")

## 5. Pareto Frontier

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot all points
for i, (name, data) in enumerate(results.items()):
    ax.scatter(data['params_m'], data['accuracy'], s=300, c=colors[i], 
               label=name, alpha=0.8, edgecolors='black', linewidth=1)
    ax.annotate(name.split('(')[0].strip(), 
                (data['params_m'], data['accuracy']),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9)

ax.set_xlabel('Parameters (M)', fontsize=14)
ax.set_ylabel('Accuracy (%)', fontsize=14)
ax.set_title('Pareto Frontier: Accuracy vs Parameters', fontsize=16)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../weights/pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nAll figures saved to weights/ folder!")